### Combine datasets from previous research 
This notebook is about downloading data

In [4]:
# Install packages
# %pip install pandas
# %pip install xlrd
# %pip install fuzzywuzzy
# %pip install pycountry
# %pip install openpyxl
# %pip install folium
# %pip install matplotlib

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [1]:
import pandas as pd
import re
import pycountry

/var/folders/h1/_zymb6k15855dxzmpdj7099w0000gp/T/ipykernel_2555/2381766060.py:1: DeprecationWarning: 
Pyarrow will become a required dependency of pandas in the next major release of pandas (pandas 3.0),
(to allow more performant data types, such as the Arrow string type, and better interoperability with other libraries)
but was not found to be installed on your system.
If this would cause problems for you,
please provide us feedback at https://github.com/pandas-dev/pandas/issues/54466
        
  import pandas as pd


In [4]:
#Load data from Sachdeva et al paper
# sachdeva_data = pd.read_csv('data/net_zero_cities_full_dataset.csv')

# sachdeva_data['plan_name'] = sachdeva_data['text'].str[:100]
# sachdeva_data_clip = sachdeva_data['name', 'iso', 'plan_name']

# sachdeva_data_clip.to_csv('sachdeva_plans.csv')


In [63]:
#Get city population data from UN World Urban Projection data
#Download Link: https://population.un.org/wup/Download/Files/WUP2018-F12-Cities_Over_300K.xls
#Latest Download Date: Feb 15, 2024
city_pop_data = pd.read_excel('data/WUP2018-F22-Cities_Over_300K_Annual.xls', skiprows=16)
#Download link: https://population.un.org/wup/Download/Files/WUP2018-F13-Capital_Cities.xls
#Latest Download Date: Feb 15, 2024
cap_city_data = pd.read_excel('data/WUP2018-F13-Capital_Cities.xls', skiprows=16)


#Using 2018 data becaause that was the last update and data for capital cities is from then
# Next update is in 2025
cities_1mill = city_pop_data[city_pop_data[2018] > 1000]

#get capital cities that have populations < 1million
cap_cities = cap_city_data[cap_city_data['Population (thousands)'] <= 1000]
# print(cities_1mill.head)
print(cap_cities.head())

   Index      Country or area  Country code      Capital City  Note  \
1      2              Albania             8   Tiranë (Tirana)   NaN   
3      4       American Samoa            16         Pago Pago   NaN   
4      5              Andorra            20  Andorra la Vella   NaN   
6      7             Anguilla           660        The Valley   NaN   
7      8  Antigua and Barbuda            28        St. John's   NaN   

  Capital Type  City code  Latitude  Longitude  Population (thousands)  \
1      Capital      20005  41.32750   19.81889                 475.577   
3      Capital      20047 -14.27806 -170.70250                  48.526   
4      Capital      20048  42.50779    1.52109                  22.614   
6      Capital      22407  18.21704  -63.05783                   1.402   
7      Capital      20052  17.11717  -61.84573                  20.764   

   Unnamed: 10  Unnamed: 11  
1          NaN          NaN  
3          NaN          NaN  
4          NaN          NaN  
6       

In [64]:
#combine datasets of cities over 1mill with capital cities
cap_cities = cap_cities.rename(columns={'Country code':'Country Code', 'City code': 'City Code'} )
city_list = pd.concat([cities_1mill, cap_cities], join='outer', ignore_index=True)
city_list['Population'] = city_list['Population (thousands)'].combine_first(city_list[2018])
city_list['City'] = city_list['Urban Agglomeration'].combine_first(city_list['Capital City'])

city_list = city_list.rename(columns={'Country or area': 'Country'} )
city_list = city_list[['Country', 'Country Code', 'City', 'City Code', 'Latitude', 'Longitude', 'Population']]

city_list['City Code'] = city_list['City Code'].astype(str)

print(city_list.head)


<bound method NDFrame.head of                           Country  Country Code                   City  \
0                     Afghanistan             4                  Kabul   
1                         Algeria            12  El Djazaïr  (Algiers)   
2                          Angola            24                 Luanda   
3                       Argentina            32           Buenos Aires   
4                       Argentina            32                Córdoba   
..                            ...           ...                    ...   
666   United Republic of Tanzania           834                 Dodoma   
667  United States Virgin Islands           850       Charlotte Amalie   
668                       Vanuatu           548              Port Vila   
669     Wallis and Futuna Islands           876               Matu-Utu   
670                Western Sahara           732               El Aaiún   

    City Code   Latitude   Longitude  Population  
0       20002  34.528887   69.

In [78]:
#Get and clean CDP 2023 Climate Actions Plan data
#Download link: https://data.cdp.net/Mitigation-Actions/2023-Cities-Climate-Action-Plans/t3a4-f7sg
#Latest Download Date: Feb 23, 2024

cdp2023_plans = pd.read_csv('data/2023_Cities_Climate_Action_Plans_20240223.csv')

cdp2023_plans_cleaned = cdp2023_plans[['Organization Number', 'Organization Name', 'City', 'Country/Area', 'Attachment/ link and name of plan', 'Year of formal approval of plan', 'City Location' ]]
cdp2023_plans_cleaned = cdp2023_plans_cleaned.rename(columns={'Organization Number': 'CDP_org_code', 'Year of formal approval of plan': 'Plan Year', 'Country/Area' : 'Country'})

#Data cleaning
#Get city name
def fill_city(row):
    if pd.notna(row['City']):
        return row['City']
    else:
        return str(row['Organization Name']).replace('City of ', '').replace('Municipality of ', '').replace('Municipality', '').replace('Municipalidad de ', '')

cdp2023_plans_cleaned['City'] = cdp2023_plans_cleaned.apply(fill_city, axis=1)

#Get link or PDF name
def extract_link(row):
    # Define a regular expression to find links
    link_pattern = r'http[s]?://(?:[a-zA-Z]|[0-9]|[$-_@.&+]|[!*\\(\\),]|(?:%[0-9a-fA-F][0-9a-fA-F]))+'
    links = re.findall(link_pattern, str(row['Attachment/ link and name of plan']))
    
    return links[0] if links else None

def extract_pdf_name(row):
    # If no links are found, try to find a PDF name using another pattern
    pdf_name_pattern = re.compile(r'\b\w+\.pdf\b', flags=re.IGNORECASE)
    pdf_names = pdf_name_pattern.findall(str(row['Attachment/ link and name of plan']))

    return pdf_names[0] if pdf_names else None

cdp2023_plans_cleaned['Plan Link'] =  cdp2023_plans_cleaned.apply(extract_link, axis=1)
cdp2023_plans_cleaned['Plan Name'] =  cdp2023_plans_cleaned.apply(extract_pdf_name, axis=1)
cdp2023_plans_cleaned['Dataset Source'] = 'CDP 2023 Climate Action Plans'
print(cdp2023_plans_cleaned)

#write cleaned up list to file
# existing_plans.to_csv('output/existing-plans-cleaned.csv')

      CDP_org_code                      Organization Name  \
0           847232                           Tres Arroyos   
1            54478               Municipality of Nijmegen   
2            54389               Taichung City Government   
3            54308  Coimbatore City Municipal Corporation   
4            54386                 Tainan City Government   
...            ...                                    ...   
1198         37261               City of Pietermaritzburg   
1199         10495                  City of Las Vegas, NV   
1200         36004                        City of Abidjan   
1201         35894                  Ville de Montreal, QC   
1202        848125                               Oreamuno   

                                       City                   Country  \
0                              Tres Arroyos                 Argentina   
1                                  Nijmegen               Netherlands   
2                                  Taichung     

In [79]:
#Get and clean data from (Olazabal et al, 2019)
olazabal_data = pd.read_excel('data/AdaptationDB.xlsx', sheet_name='AdaptationDB')

#Get only municipal-level plans
olazabal_data = olazabal_data[olazabal_data['Policy scale'].isin(['Metropolitan', 'Local', 'Metropolitan area', 'Metropilitan/Regional', 'Local & Regional', 'National - Local', 'City' ])]

#Select important columns & rename
olazabal_data = olazabal_data[['Name of the policy', 'City', 'Country', 'Language of the document', 'Year of publication', 
                               'Link to the document', 'Contact administration ', 'Contact administration website']]

olazabal_data = olazabal_data.rename(columns={'Name of the policy': 'Plan Name', 'Language of the document': 'Document Language', 'Year of publication' : 'Plan Year', 
                                              'Link to the document': 'Plan Link', 'Contact administration ': 'Organization Name', 'Contact administration website': 'Municipal Link'})

olazabal_data['Dataset Source'] = 'Olazabal et al., "A cross-scale worldwide analysis of coastal adaptation planning"'
print(olazabal_data.head())

                                            Plan Name       City  \
0   City of Cape Town Climate Change Policy (Polic...  Cape Town   
1   Integrated Coastal Management Policy of the Ci...  Cape Town   
6                      Durban Climate Change Strategy     Durban   
14                          Dakar Resilience Strategy      Dakar   
26  Helsinki Metropolitan Area Climate Change Adap...   Helsinki   

         Country Document Language Plan Year  \
0   South Africa           English      2017   
1   South Africa           English      2014   
6   South Africa           English      2014   
14       Senegal           English      2017   
26       Finland           English      2012   

                                            Plan Link  \
0   https://savingelectricity.org.za/wp-content/up...   
1   https://www.google.com/url?sa=t&rct=j&q=&esrc=...   
6   http://www.durban.gov.za/City_Services/energyo...   
14  https://www.100resilientcities.org/wp-content/...   
26  http://ilmast

In [80]:
#Merge existing lists of plans with CDP data
#Get existing list of plans
manual_existing_plans = pd.read_csv('data/Data - List of Existing Cities_Plans.csv')
existing_plans = pd.concat([manual_existing_plans, cdp2023_plans_cleaned], join='outer', ignore_index=True)
existing_plans = pd.concat([existing_plans, olazabal_data], join='outer', ignore_index=True)

#write existing plans to file
existing_plans.to_csv('output/combined_existing_plans.csv')
print(existing_plans)

           City      Country  \
0     Amsterdam  Netherlands   
1     Amsterdam  Netherlands   
2     Amsterdam  Netherlands   
3     Amsterdam  Netherlands   
4     Amsterdam  Netherlands   
...         ...          ...   
1722    Incheon  South Korea   
1723      Busan  South Korea   
1724      Ulsan  South Korea   
1725   Auckland  New Zealand   
1726   Auckland  New Zealand   

                                              Plan Name  \
0                  Amsterdam: a different energy (SEAP)   
1                      Amsterdam definitely sustainable   
2                                 New Amsterdam climate   
3              Outspokenly sustainable-perspective 2014   
4                   Structure vision for Amsterdam 2014   
...                                                 ...   
1722  Second Incheon Metropolitan City Detailed Plan...   
1723  Second Busan Metropolitan City Detailed Plan f...   
1724  Second Ulsan Metropolitan City Detailed Plan f...   
1725       New Zealand Co

In [81]:
# Get ISO codes

# Function to get ISO code for a given country name
def get_iso_code(country_name):
    country_name = str(country_name)
    #Manual add in for missing ISO codes because search_fuzzy doesn't seem to work well for some reason
    country_codes = {"Bolivia (Plurinational State of)": "BOL", "China, Taiwan Province of China": "TWN", 
                     "China, Hong Kong SAR": "HKG", "Democratic Republic of the Congo" : "COD", "Iran (Islamic Republic of)": "IRN", "Turkey": "TUR", 
                     "Venezuela (Bolivarian Republic of)" : "VEN", "Micronesia (Fed. States of)": "FSM", "United States Virgin Islands" : "VIR", "Wallis and Futuna Islands": "WLF", 
                     "TFYR Macedonia" : "MKD", "Swaziland": "SWZ", "Faeroe Islands" : "FRO", "China, Macao SAR": "MAC", "Channel Islands" : "GBR", "Dem. People's Republic of Korea" : "PRK"}
        
    if country_name in country_codes.keys():
        return country_codes[country_name]
    try:
        # Use pycountry to get ISO code
        country = pycountry.countries.search_fuzzy(country_name)[0]
        return country.alpha_3
    except LookupError:
        print(country_name)
        # Return None if the country name is not found
        return None
    
 
existing_plans['ISO_Code'] = existing_plans['Country'].apply(get_iso_code)
city_list['ISO_Code'] = city_list['Country'].apply(get_iso_code)

SubdivisionHierarchy(code='TH-55', country_code='TH', name='Nan', parent_code=None, type='Province')
SubdivisionHierarchy(code='US-AK', country_code='US', name='Alaska', parent_code=None, type='State')
SubdivisionHierarchy(code='US-AL', country_code='US', name='Alabama', parent_code=None, type='State')
SubdivisionHierarchy(code='US-AR', country_code='US', name='Arkansas', parent_code=None, type='State')
SubdivisionHierarchy(code='US-AS', country_code='US', name='American Samoa', parent_code=None, type='Outlying area')
SubdivisionHierarchy(code='US-AZ', country_code='US', name='Arizona', parent_code=None, type='State')
SubdivisionHierarchy(code='US-CA', country_code='US', name='California', parent_code=None, type='State')
SubdivisionHierarchy(code='US-CO', country_code='US', name='Colorado', parent_code=None, type='State')
SubdivisionHierarchy(code='US-CT', country_code='US', name='Connecticut', parent_code=None, type='State')
SubdivisionHierarchy(code='US-DC', country_code='US', name='

In [76]:
# Get main language for each city

#Function to get all languages for each country
def get_languages(country_code):
    try:
        country_langs = country_info[country_info['ISO3'] == country_code]['Languages']
        if not country_langs.empty:
            country_langs = country_langs.iloc[0].split(',')
            main_lang = country_langs[0].split('-')[0]
            other_langs = country_langs[1:]
            return main_lang, other_langs
    except Exception as e:
        print(f"Error: {e}")
        return None
    
#Function to get language name (in English) from language code    
def get_language_name(lang_code):
    try:
        language = pycountry.languages.get(alpha_2=lang_code)
        if language:
            return language.name
    except Exception as e:
        return None    

#Country Info txt file comes from http://download.geonames.org/export/dump/countryInfo.txt
country_info = pd.read_csv('data/countryInfo.txt', sep='\t')
city_list[['Language_Code', 'Other Languages']] = city_list['ISO_Code'].apply(get_languages).apply(pd.Series)
city_list['Language'] = city_list['Language_Code'].apply(get_language_name)
city_list


,Country,Country Code,City,City Code,Latitude,Longitude,Population,ISO_Code,Language_Code,Other Languages,Language
0,Afghanistan,4,Kabul,20002,34.528887,69.172460,4011.770,AFG,fa,"[ps, uz-AF, tk]",Persian
1,Algeria,12,El Djazaïr (Algiers),20006,36.752500,3.041970,2693.542,DZA,ar,[],Arabic
2,Angola,24,Luanda,20049,-8.836820,13.234320,7774.200,AGO,pt,[],Portuguese
3,Argentina,32,Buenos Aires,20058,-34.605083,-58.400368,14966.530,ARG,es,"[en, it, de, fr, gn]",Spanish
4,Argentina,32,Córdoba,20059,-31.413500,-64.181050,1548.176,ARG,es,"[en, it, de, fr, gn]",Spanish
...,...,...,...,...,...,...,...,...,...,...,...
666,United Republic of Tanzania,834,Dodoma,22895,-6.172210,35.739470,261.645,TZA,sw,"[en, ar]",Swahili (macrolanguage)
667,United States Virgin Islands,850,Charlotte Amalie,23190,18.341900,-64.930700,52.322,VIR,en,[],English
668,Vanuatu,548,Port Vila,21956,-17.733810,168.321880,52.690,VUT,bi,"[en-VU, fr-VU]",Bislama
669,Wallis and Futuna Islands,876,Matu-Utu,23235,-13.281630,-176.174530,1.025,WLF,wls,"[fud, fr-WF]",None


In [77]:
#write list to file
city_list.to_csv('output/capital_and_1mill_pop_cities.csv')

In [12]:
#Find matches between cities with existing plans and the desired list of cities
def find_matching_city(row, city_list_group):

    # Check if the city names match or if one is contained in the other
    matches = city_list_group['City'].apply(lambda city: city in row['City'] or row['City'] in city)
    matching_city_codes = city_list_group.loc[matches, 'City Code'].tolist()

    if matching_city_codes:
        return ', '.join(matching_city_codes)
    else:
        return None
    
def get_country_group(row, city_list):
    try: 
        return find_matching_city(row, city_list.groupby('ISO_Code').get_group(row['ISO_Code']))
    except KeyError:
        pass
    return None

existing_plans['matched_cities'] = existing_plans.apply(get_country_group,city_list=city_list,axis=1)
print(existing_plans)

           City      Country  \
0     Amsterdam  Netherlands   
1     Amsterdam  Netherlands   
2     Amsterdam  Netherlands   
3     Amsterdam  Netherlands   
4     Amsterdam  Netherlands   
...         ...          ...   
1722    Incheon  South Korea   
1723      Busan  South Korea   
1724      Ulsan  South Korea   
1725   Auckland  New Zealand   
1726   Auckland  New Zealand   

                                              Plan Name  \
0                  Amsterdam: a different energy (SEAP)   
1                      Amsterdam definitely sustainable   
2                                 New Amsterdam climate   
3              Outspokenly sustainable-perspective 2014   
4                   Structure vision for Amsterdam 2014   
...                                                 ...   
1722  Second Incheon Metropolitan City Detailed Plan...   
1723  Second Busan Metropolitan City Detailed Plan f...   
1724  Second Ulsan Metropolitan City Detailed Plan f...   
1725       New Zealand Co

In [13]:
#Merge the datasets 
existing_plans = existing_plans.drop(columns=['City', 'Country', 'ISO_Code'])
merged_dataset = city_list.merge(existing_plans, left_on='City Code', right_on='matched_cities',  how='left')

#This is if we want only the cities with existing plans (doesn't include the full cities list)
# merged_dataset = existing_plans.merge(city_list, left_on='matched_cities', right_on='city_code', how='left')

#Cleanup
merged_dataset = merged_dataset.drop(columns=['Country Code', 'matched_cities', 'City Location', 'CDP_org_code'])
print(merged_dataset)

#write cleaned up list to file
merged_dataset.to_csv('output/merged-data.csv')

                          Country                   City City Code   Latitude  \
0                     Afghanistan                  Kabul     20002  34.528887   
1                         Algeria  El Djazaïr  (Algiers)     20006  36.752500   
2                          Angola                 Luanda     20049  -8.836820   
3                       Argentina           Buenos Aires     20058 -34.605083   
4                       Argentina           Buenos Aires     20058 -34.605083   
..                            ...                    ...       ...        ...   
978   United Republic of Tanzania                 Dodoma     22895  -6.172210   
979  United States Virgin Islands       Charlotte Amalie     23190  18.341900   
980                       Vanuatu              Port Vila     21956 -17.733810   
981     Wallis and Futuna Islands               Matu-Utu     23235 -13.281630   
982                Western Sahara               El Aaiún     22571  27.153164   

      Longitude  Population

### Stats on Data

In [41]:
import folium
import pandas as pd
from IPython.display import display

# Group by latitude and longitude, and check if any row has a 'Plan Link'
grouped_data = merged_dataset.groupby(['Latitude', 'Longitude']).agg({
    'City': 'first',
    'Plan Link': lambda x: any(x.notnull())
}).reset_index()

# Create a folium map centered around the mean coordinates
map_center = [grouped_data['Latitude'].mean(), grouped_data['Longitude'].mean()]
my_map = folium.Map(location=map_center, zoom_start=4)

# Add markers for each data point
for index, row in grouped_data.iterrows():
    marker_color = 'green' if row['Plan Link'] else 'blue'
    folium.Marker(location=[row['Latitude'], row['Longitude']], popup=row['City'], icon=folium.Icon(color=marker_color, icon='fa-solid fa-circle')).add_to(my_map)

# Display the map directly in the Jupyter Notebook
display(my_map)

my_map.save('output/city_plans_map.html')


In [34]:
# Drop duplicate rows based on 'City', 'Latitude', and 'Longitude'
unique_locations = merged_dataset.drop_duplicates(subset=['City', 'Latitude', 'Longitude'])

# Count the number of unique locations with plans and without plans
locations_with_plans = unique_locations[unique_locations['Plan Link'].notna()]
locations_without_plans = unique_locations[unique_locations['Plan Link'].isna()]

print(f"1) Number of unique locations with plans: {len(locations_with_plans)}")
print(f"   Number of unique locations without plans: {len(locations_without_plans)}")

# Group by country and count the number of unique locations with plans for each country
locations_with_plans_by_country = locations_with_plans.groupby('Country').size().reset_index(name='LocationsWithPlans')
countries_without_plans = locations_without_plans['Country'].unique()

print("\n2) Countries with the most unique locations with plans:")
print(locations_with_plans_by_country.sort_values(by='LocationsWithPlans', ascending=False).head())

print("\n   Countries without unique locations with plans:")
print(countries_without_plans)

# Identify unique locations with plans and multiple years
locations_with_multiple_years = merged_dataset.groupby(['City', 'Latitude', 'Longitude']).filter(lambda x: x['Plan Year'].nunique() > 1)
print("\n3) Cities with plans with multiple years:")
print(locations_with_multiple_years.drop_duplicates(subset=['City', 'Latitude', 'Longitude']).count())
print(locations_with_multiple_years[['Country', 'City', 'Plan Year', 'Plan Link']])
locations_with_multiple_years.to_csv('output/multiyear_plans.csv')



1) Number of unique locations with plans: 98
   Number of unique locations without plans: 573

2) Countries with the most unique locations with plans:
                     Country  LocationsWithPlans
28  United States of America                  26
6                      China                  12
3                     Brazil                   6
4                     Canada                   6
13                     Japan                   6

   Countries without unique locations with plans:
['Afghanistan' 'Algeria' 'Angola' 'Argentina' 'Armenia' 'Australia'
 'Austria' 'Azerbaijan' 'Bangladesh' 'Belarus' 'Belgium'
 'Bolivia (Plurinational State of)' 'Brazil' 'Bulgaria' 'Burkina Faso'
 'Cambodia' 'Cameroon' 'Chad' 'China' 'China, Hong Kong SAR'
 'China, Taiwan Province of China' 'Colombia' 'Congo' 'Costa Rica'
 "Côte d'Ivoire" 'Cuba' 'Czechia' "Dem. People's Republic of Korea"
 'Democratic Republic of the Congo' 'Denmark' 'Dominican Republic'
 'Ecuador' 'Egypt' 'El Salvador' 'Ethiopia' '